# OC20Dense accuracy and reproducibility check

Use this notebook to double-check the closed-shell OC20Dense benchmark interactively. It reloads generated outputs for provenance checks, recomputes ranking metrics and adsorption-energy subtraction, verifies DFT reference arithmetic, plots MACE-vs-DFT adsorption-energy parity, and opens selected structures with OVITO widgets.

This benchmark slice is intentionally narrow: H2O, NH3, and N2 on OC20Dense slabs. It is separate from the active CO/H2O/CH3OH teaching panel in the main notebook. The point is not to claim that three systems represent all adsorption chemistry; the point is to show a reproducible reference-backed check where the DFT trajectory, clean-slab reference, gas reference, MACE single point, and Toolkit relaxation can all be audited.

For a stronger check, use the **Live subset recomputation** section. It reruns a small set of exact OC20Dense configs through the Toolkit relaxation, DFT trajectory check, DFT-relaxed final single point, and MACE adsorption-energy subtraction, then compares the fresh outputs with the saved full-run tables.


## What this checks

- The closed-shell benchmark slice is `*OH2`/H2O, `*NH3`/NH3, and `*N2`/N2, chosen because the neutral gas references are unambiguous for this tutorial check.
- The DFT trajectory final energy reproduces the released OC20Dense adsorption-energy target exactly.
- Initial single points, DFT-relaxed final single points, and Toolkit relaxations are recomputed from source tables, not from the markdown report.
- MACE adsorption energies are recomputed from `E_ads = E(adslab) - E(surface) - E(gas)` using the saved MACE reference terms.
- D3 is available in Toolkit workflows, but it is not used in this reference check because the released OC20Dense reference targets used here do not include a D3 correction.
- Live subset recomputation reruns selected configs from source structures, which is the real reproducibility check.
- OVITO widgets and static render panels let you inspect initial, official DFT-relaxed final, and Toolkit-relaxed geometries side by side.


In [ ]:
from pathlib import Path
import math
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display


def find_part1() -> Path:
    here = Path.cwd().resolve()
    if (here / "helpers" / "__init__.py").exists():
        return here
    candidate = here / "part-1-batched-adsorption"
    if (candidate / "helpers" / "__init__.py").exists():
        return candidate
    raise FileNotFoundError(
        "Run this notebook from the tutorial directory or from the repository root."
    )


PART1 = find_part1()
if str(PART1) not in sys.path:
    sys.path.insert(0, str(PART1))

from helpers.references import LITERATURE_OC157_MAD_GUIDE_EV

ROOT = PART1 / "outputs" / "oc20dense_known_examples"
EXPECTED_SYSTEMS = {
    "3_2070_48": ("*OH2", "H2O"),
    "72_7104_115": ("*NH3", "NH3"),
    "69_1615_2": ("*N2", "N2"),
}
SUCCESS_GAP_EV = 0.10

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 6)

print(f"PART1 = {PART1}")
print(f"ROOT  = {ROOT}")


In [ ]:
TABLES = {
    "system_summary": ROOT / "tables" / "system_summary.csv",
    "per_config": ROOT / "tables" / "per_config_results.csv",
    "accuracy_layer": ROOT / "tables" / "accuracy_layer_summary.csv",
    "accuracy_aggregate": ROOT / "tables" / "accuracy_aggregate_summary.csv",
    "dft_reference": ROOT / "dft_reference_checks" / "dft_reference_comparison.csv",
    "selected_cases": ROOT / "dft_reference_checks" / "selected_case_comparison.csv",
    "dft_final_sp": ROOT / "dft_final_single_points" / "tables" / "dft_final_sp_results.csv",
    "dft_final_sp_summary": ROOT / "dft_final_single_points" / "tables" / "dft_final_sp_system_summary.csv",
    "mace_eads": ROOT / "mace_adsorption_energy" / "tables" / "mace_adsorption_energies.csv",
    "mace_eads_summary": ROOT / "mace_adsorption_energy" / "tables" / "mace_adsorption_energy_summary.csv",
    "mace_refs": ROOT / "mace_adsorption_energy" / "tables" / "mace_adsorption_reference_energies.csv",
}

path_check = pd.DataFrame(
    {"table": name, "path": str(path), "exists": path.exists()} for name, path in TABLES.items()
)
display(path_check)
missing = path_check.loc[~path_check["exists"], "path"].tolist()
assert not missing, "Missing required benchmark outputs:\n" + "\n".join(missing)

In [ ]:
tables = {name: pd.read_csv(path) for name, path in TABLES.items()}

system_summary = tables["system_summary"]
per_config = tables["per_config"]
accuracy_layer = tables["accuracy_layer"]
accuracy_aggregate = tables["accuracy_aggregate"]
dft_reference = tables["dft_reference"]
dft_final_sp = tables["dft_final_sp"]
eads = tables["mace_eads"]
eads_summary = tables["mace_eads_summary"]
mace_refs = tables["mace_refs"]

observed = {
    row["system_id"]: (row["adsorbate"], row["adsorbate_reference_species"])
    for _, row in system_summary.iterrows()
}
assert observed == EXPECTED_SYSTEMS, observed
assert set(per_config["system_id"]) == set(EXPECTED_SYSTEMS)
assert set(dft_reference["system_id"]) == set(EXPECTED_SYSTEMS)
assert not per_config["adsorbate"].str.contains("COH|CH3", regex=True).any()
assert not dft_final_sp["adsorbate"].str.contains("COH|CH3", regex=True).any()

status_values = sorted(
    set(per_config["mace_eads_reference_status"])
    | set(dft_reference["mace_eads_reference_status"])
    | set(dft_final_sp["mace_eads_reference_status"])
    | set(eads["mace_eads_reference_status"])
)
assert status_values == ["defined_mace_eads_official_surface_neutral_gas_refs"], status_values

display(Markdown("### Closed-shell benchmark slice"))
display(system_summary[["system_id", "adsorbate", "adsorbate_reference_species", "n_configs", "mace_rank_basis", "mace_eads_reference_status"]])

In [ ]:
numeric_cols = [
    "dft_traj_minus_target_eV",
    "start_active_atom_rmsd_A",
    "start_adsorbate_rmsd_A",
    "mic_active_atom_rmsd_A",
    "mic_adsorbate_rmsd_A",
]
for col in numeric_cols:
    dft_reference[col] = pd.to_numeric(dft_reference[col])

integrity = pd.DataFrame(
    [
        {
            "exact_trajectory_records": len(dft_reference),
            "max_abs_dft_traj_minus_target_eV": dft_reference["dft_traj_minus_target_eV"].abs().max(),
            "max_start_active_rmsd_A": dft_reference["start_active_atom_rmsd_A"].max(),
            "max_start_adsorbate_rmsd_A": dft_reference["start_adsorbate_rmsd_A"].max(),
            "max_final_mic_active_rmsd_A": dft_reference["mic_active_atom_rmsd_A"].max(),
            "max_final_mic_adsorbate_rmsd_A": dft_reference["mic_adsorbate_rmsd_A"].max(),
        }
    ]
)

display(Markdown("### DFT reference integrity"))
display(integrity)
display(dft_reference.groupby(["system_id", "adsorbate", "adsorbate_reference_species"]).size().rename("n_records").reset_index())
assert integrity.loc[0, "max_abs_dft_traj_minus_target_eV"] <= 1e-12

In [ ]:
def recompute_ranking(df: pd.DataFrame, energy_col: str, label: str) -> pd.DataFrame:
    rows = []
    for system_id, group in df.groupby("system_id", sort=False):
        group = group.copy()
        group["dft_adsorption_energy_eV"] = pd.to_numeric(group["dft_adsorption_energy_eV"])
        group[energy_col] = pd.to_numeric(group[energy_col])
        group["dft_rank"] = pd.to_numeric(group["dft_rank"])
        group["dft_gap_recomputed_eV"] = group["dft_adsorption_energy_eV"] - group["dft_adsorption_energy_eV"].min()

        ranked = group.sort_values(energy_col, kind="mergesort")
        top1 = ranked.iloc[0]
        top3 = ranked.head(min(3, len(ranked)))
        top5 = ranked.head(min(5, len(ranked)))
        ml_rank = group[energy_col].rank(method="min", ascending=True)
        spearman = group["dft_rank"].corr(ml_rank, method="spearman")

        rows.append(
            {
                "layer": label,
                "system_id": system_id,
                "adsorbate": top1["adsorbate"],
                "adsorbate_reference_species": top1["adsorbate_reference_species"],
                "n_configs": len(group),
                "ml_best_config": top1["config_id"],
                "ml_best_dft_rank": int(top1["dft_rank"]),
                "top1_gap_eV": float(top1["dft_gap_recomputed_eV"]),
                "top3_best_gap_eV": float(top3["dft_gap_recomputed_eV"].min()),
                "top5_best_gap_eV": float(top5["dft_gap_recomputed_eV"].min()),
                "top1_success_0p10eV": bool(top1["dft_gap_recomputed_eV"] <= SUCCESS_GAP_EV),
                "top3_success_0p10eV": bool(top3["dft_gap_recomputed_eV"].min() <= SUCCESS_GAP_EV),
                "spearman": float(spearman) if pd.notna(spearman) else np.nan,
            }
        )
    return pd.DataFrame(rows)

ranking_recomputed = pd.concat(
    [
        recompute_ranking(per_config, "ml_initial_sp_total_energy_eV", "initial-coordinate SP"),
        recompute_ranking(dft_final_sp, "mace_dft_final_sp_total_energy_eV", "DFT-relaxed final SP"),
        recompute_ranking(per_config, "ml_total_energy_eV", "Toolkit relaxation"),
    ],
    ignore_index=True,
)

aggregate_recomputed = (
    ranking_recomputed.groupby("layer", sort=False)
    .agg(
        top1_success_0p10eV=("top1_success_0p10eV", lambda x: f"{int(x.sum())}/{len(x)}"),
        top3_success_0p10eV=("top3_success_0p10eV", lambda x: f"{int(x.sum())}/{len(x)}"),
        median_top1_gap_eV=("top1_gap_eV", "median"),
        max_top1_gap_eV=("top1_gap_eV", "max"),
        median_spearman=("spearman", "median"),
    )
    .reset_index()
)

display(Markdown("### Recomputed ranking accuracy"))
display(aggregate_recomputed)
display(ranking_recomputed.sort_values(["system_id", "layer"]))

reported = accuracy_aggregate[["layer", "top1_success_0p10eV", "top3_success_0p10eV"]]
check = aggregate_recomputed.merge(reported, on="layer", suffixes=("_recomputed", "_reported"))
assert (check["top1_success_0p10eV_recomputed"] == check["top1_success_0p10eV_reported"]).all()
assert (check["top3_success_0p10eV_recomputed"] == check["top3_success_0p10eV_reported"]).all()

In [ ]:
eads = eads.copy()
for col in [
    "dft_adsorption_energy_target_eV",
    "mace_dft_final_sp_total_energy_eV",
    "ml_total_energy_eV",
    "mace_surface_dft_final_sp_energy_eV",
    "mace_surface_relaxed_energy_eV",
    "mace_gas_energy_eV",
    "mace_dft_final_eads_eV",
    "mace_relaxed_eads_eV",
]:
    eads[col] = pd.to_numeric(eads[col])

eads["recalc_mace_dft_final_eads_eV"] = (
    eads["mace_dft_final_sp_total_energy_eV"]
    - eads["mace_surface_dft_final_sp_energy_eV"]
    - eads["mace_gas_energy_eV"]
)
eads["recalc_mace_relaxed_eads_eV"] = (
    eads["ml_total_energy_eV"]
    - eads["mace_surface_relaxed_energy_eV"]
    - eads["mace_gas_energy_eV"]
)
eads["recalc_dft_final_error_eV"] = eads["recalc_mace_dft_final_eads_eV"] - eads["dft_adsorption_energy_target_eV"]
eads["recalc_relaxed_error_eV"] = eads["recalc_mace_relaxed_eads_eV"] - eads["dft_adsorption_energy_target_eV"]

max_eads_roundtrip_error = max(
    (eads["recalc_mace_dft_final_eads_eV"] - eads["mace_dft_final_eads_eV"]).abs().max(),
    (eads["recalc_mace_relaxed_eads_eV"] - eads["mace_relaxed_eads_eV"]).abs().max(),
)
assert max_eads_roundtrip_error <= 1e-10


def summarize_eads(group: pd.DataFrame) -> pd.Series:
    dft_final_err = group["recalc_dft_final_error_eV"]
    relaxed_err = group["recalc_relaxed_error_eV"]
    return pd.Series(
        {
            "n_configs": len(group),
            "dft_final_eads_mae_eV": dft_final_err.abs().mean(),
            "dft_final_eads_bias_eV": dft_final_err.mean(),
            "dft_final_eads_rmse_eV": math.sqrt(float((dft_final_err ** 2).mean())),
            "relaxed_eads_mae_eV": relaxed_err.abs().mean(),
            "relaxed_eads_bias_eV": relaxed_err.mean(),
            "relaxed_eads_rmse_eV": math.sqrt(float((relaxed_err ** 2).mean())),
        }
    )

recomputed_eads_summary = (
    eads.groupby(["system_id", "adsorbate", "adsorbate_reference_species"], sort=False)[
        ["recalc_dft_final_error_eV", "recalc_relaxed_error_eV"]
    ]
    .apply(summarize_eads)
    .reset_index()
)

display(Markdown("### Recomputed MACE adsorption energies"))
display(recomputed_eads_summary)
display(Markdown(f"Max Eads table round-trip error: `{max_eads_roundtrip_error:.3e} eV`"))
display(Markdown("Reference energies used in the subtraction"))
display(mace_refs[["system_id", "adsorbate", "adsorbate_reference_species", "mace_surface_dft_final_sp_energy_eV", "mace_surface_relaxed_energy_eV", "mace_gas_energy_eV", "surface_relaxed_converged", "gas_converged"]])

In [ ]:
import matplotlib.pyplot as plt

PLOTS = ROOT / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)
PARITY_FIGURE = PLOTS / "oc20dense_mace_eads_parity.png"

plot_df = eads.copy()
fig, (ax_scatter, ax_hist) = plt.subplots(1, 2, figsize=(11.8, 4.8), gridspec_kw={"width_ratios": [1.15, 0.85]})
colors = {"*OH2": "#1f77b4", "*NH3": "#2ca02c", "*N2": "#9467bd"}
markers = {"DFT-relaxed final geometry SP": "o", "Toolkit-relaxed geometry": "^"}

x = plot_df["dft_adsorption_energy_target_eV"].astype(float)
series = {
    "DFT-relaxed final geometry SP": plot_df["recalc_mace_dft_final_eads_eV"].astype(float),
    "Toolkit-relaxed geometry": plot_df["recalc_mace_relaxed_eads_eV"].astype(float),
}
limits = [min(x.min(), *(y.min() for y in series.values())) - 0.2,
          max(x.max(), *(y.max() for y in series.values())) + 0.2]
ax_scatter.plot(limits, limits, color="black", linewidth=1.1, label="perfect parity")
ax_scatter.fill_between(
    limits,
    [v - LITERATURE_OC157_MAD_GUIDE_EV for v in limits],
    [v + LITERATURE_OC157_MAD_GUIDE_EV for v in limits],
    color="#76B900",
    alpha=0.14,
    label=f"+/- {LITERATURE_OC157_MAD_GUIDE_EV:.2f} eV literature MAD guide",
)
for adsorbate, group in plot_df.groupby("adsorbate", sort=False):
    idx = group.index
    for label, y in series.items():
        ax_scatter.scatter(
            x.loc[idx],
            y.loc[idx],
            marker=markers[label],
            s=42,
            alpha=0.72,
            color=colors.get(adsorbate, "#555555"),
            edgecolor="white",
            linewidth=0.35,
            label=f"{adsorbate} | {label}",
        )
for adsorbate, group in plot_df.groupby("adsorbate", sort=False):
    idx = group.index
    residual_values = np.concatenate([(y.loc[idx] - x.loc[idx]).to_numpy() for y in series.values()])
    bias = float(np.mean(residual_values))
    if abs(bias) < 0.15:
        bias_label = f"{adsorbate} near parity"
    else:
        direction = "underbinds" if bias > 0 else "overbinds"
        bias_label = f"{adsorbate} {direction} ~{abs(bias):.2f} eV"
    y_values = np.concatenate([y.loc[idx].to_numpy() for y in series.values()])
    ax_scatter.annotate(
        bias_label,
        xy=(float(x.loc[idx].mean()), float(np.mean(y_values))),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=7,
        color=colors.get(adsorbate, "#555555"),
    )
ax_scatter.set_xlim(limits)
ax_scatter.set_ylim(limits)
ax_scatter.set_aspect("equal", adjustable="box")
ax_scatter.set_xlabel("DFT adsorption energy target (eV)")
ax_scatter.set_ylabel("MACE adsorption energy, recomputed (eV)")
ax_scatter.set_title("Adsorption-energy parity")
ax_scatter.grid(True, linestyle="--", alpha=0.35)
ax_scatter.legend(fontsize=6, loc="lower right", frameon=False)

residuals = pd.DataFrame(
    {
        "DFT-relaxed final geometry SP": series["DFT-relaxed final geometry SP"] - x,
        "Toolkit-relaxed geometry": series["Toolkit-relaxed geometry"] - x,
    }
)
bins = np.linspace(residuals.min().min() - 0.05, residuals.max().max() + 0.05, 22)
ax_hist.hist(residuals["DFT-relaxed final geometry SP"], bins=bins, alpha=0.66, color="#00A3E0", label="DFT-relaxed final SP")
ax_hist.hist(residuals["Toolkit-relaxed geometry"], bins=bins, alpha=0.54, color="#76B900", label="Toolkit relaxed")
ax_hist.axvline(0.0, color="black", linewidth=1.0)
ax_hist.axvline(LITERATURE_OC157_MAD_GUIDE_EV, color="#76B900", linestyle="--", linewidth=1.0)
ax_hist.axvline(-LITERATURE_OC157_MAD_GUIDE_EV, color="#76B900", linestyle="--", linewidth=1.0)
ax_hist.set_xlabel("MACE - DFT adsorption energy (eV)")
ax_hist.set_ylabel("Configurations")
ax_hist.set_title("Residuals")
ax_hist.grid(True, axis="y", linestyle="--", alpha=0.35)
ax_hist.legend(fontsize=8, frameon=False)

fig.suptitle("OC20Dense closed-shell slice: exact DFT targets vs MACE-defined adsorption energies", y=1.03)
fig.tight_layout()
fig.savefig(PARITY_FIGURE, dpi=180, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(PARITY_FIGURE)))
print(f"Saved: {PARITY_FIGURE}")


In [ ]:
path_cols = ["system_id", "config_id", "sid", "dft_final_structure_path", "dft_trajectory_path"]
rmsd_cols = ["system_id", "config_id", "sid", "mic_active_atom_rmsd_A", "mic_adsorbate_rmsd_A", "raw_active_atom_rmsd_A", "raw_adsorbate_rmsd_A"]

case_table = per_config.merge(
    dft_final_sp[path_cols],
    on=["system_id", "config_id", "sid"],
    how="left",
).merge(
    dft_reference[rmsd_cols],
    on=["system_id", "config_id", "sid"],
    how="left",
)

CASE_OPTIONS = [
    "DFT best",
    "Initial SP best",
    "DFT-relaxed final SP best",
    "Toolkit-relaxed best",
    "Largest MIC adsorbate RMSD",
]


def pick_case(system_id: str, case_name: str) -> pd.Series:
    group = case_table.loc[case_table["system_id"] == system_id].copy()
    if group.empty:
        raise ValueError(f"Unknown system_id: {system_id}")
    if case_name == "DFT best":
        idx = group["dft_adsorption_energy_eV"].astype(float).idxmin()
    elif case_name == "Initial SP best":
        idx = group["ml_initial_sp_total_energy_eV"].astype(float).idxmin()
    elif case_name == "DFT-relaxed final SP best":
        idx = group["mace_dft_final_sp_total_energy_eV"].astype(float).idxmin()
    elif case_name == "Toolkit-relaxed best":
        idx = group["ml_total_energy_eV"].astype(float).idxmin()
    elif case_name == "Largest MIC adsorbate RMSD":
        idx = group["mic_adsorbate_rmsd_A"].astype(float).idxmax()
    else:
        raise ValueError(case_name)
    return group.loc[idx]


def case_summary(row: pd.Series) -> pd.DataFrame:
    path_fields = ["initial_structure", "dft_final_structure_path", "relaxed_structure", "raw_json"]
    summary = {
        "system_id": row["system_id"],
        "config_id": row["config_id"],
        "sid": int(row["sid"]),
        "adsorbate": row["adsorbate"],
        "reference_species": row["adsorbate_reference_species"],
        "dft_rank": int(row["dft_rank"]),
        "dft_adsorption_energy_eV": float(row["dft_adsorption_energy_eV"]),
        "dft_gap_to_best_eV": float(row["dft_gap_to_best_eV"]),
        "initial_sp_rank": int(row["ml_initial_sp_rank"]),
        "relaxed_rank": int(row["ml_relaxed_rank"]),
        "mic_active_rmsd_A": float(row["mic_active_atom_rmsd_A"]),
        "mic_adsorbate_rmsd_A": float(row["mic_adsorbate_rmsd_A"]),
    }
    for field in path_fields:
        summary[f"{field}_exists"] = Path(str(row[field])).exists()
    return pd.DataFrame([summary])

row = pick_case("3_2070_48", "DFT best")
display(case_summary(row))

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

system_dropdown = widgets.Dropdown(
    options=[(f"{sid} {ads} -> {ref}", sid) for sid, (ads, ref) in EXPECTED_SYSTEMS.items()],
    value="3_2070_48",
    description="System",
    layout=widgets.Layout(width="420px"),
)
case_dropdown = widgets.Dropdown(
    options=CASE_OPTIONS,
    value="DFT best",
    description="Case",
    layout=widgets.Layout(width="320px"),
)
case_output = widgets.Output()
CURRENT_CASE = None


def refresh_case(*_):
    global CURRENT_CASE
    with case_output:
        clear_output(wait=True)
        CURRENT_CASE = pick_case(system_dropdown.value, case_dropdown.value)
        display(case_summary(CURRENT_CASE))
        display(
            pd.DataFrame(
                [
                    {"structure": "initial", "path": CURRENT_CASE["initial_structure"]},
                    {"structure": "official_dft_final", "path": CURRENT_CASE["dft_final_structure_path"]},
                    {"structure": "toolkit_relaxed", "path": CURRENT_CASE["relaxed_structure"]},
                ]
            )
        )

system_dropdown.observe(refresh_case, names="value")
case_dropdown.observe(refresh_case, names="value")
refresh_case()
display(widgets.HBox([system_dropdown, case_dropdown]), case_output)

## Live subset recomputation

The cells above check the generated artifacts. This section reruns a small, explicit subset from the source structures and compares the fresh outputs with the saved benchmark tables.

Start with two or three configs. Keeping `max optimizer steps = 200` gives the apples-to-apples setting used in the full benchmark; lowering it is useful only for debugging runtime.

This is the best place to audit the claim that the workflow is reproducible: choose exact configs, rerun the pipeline, and inspect the fresh deltas.


In [ ]:
import datetime as dt
import shlex
import subprocess

SUBSET_BASE = PART1 / "outputs" / "oc20dense_subset_recompute"


def config_options_for_system(system_id: str) -> list[tuple[str, str]]:
    group = case_table.loc[case_table["system_id"] == system_id].copy()
    group["dft_rank"] = group["dft_rank"].astype(int)
    group["dft_gap_to_best_eV"] = group["dft_gap_to_best_eV"].astype(float)
    group = group.sort_values(["dft_rank", "config_id"])
    options = []
    for _, row in group.iterrows():
        label = (
            f"{row['config_id']} | sid {int(row['sid'])} | "
            f"DFT rank {int(row['dft_rank'])} | gap {float(row['dft_gap_to_best_eV']):.4f} eV"
        )
        options.append((label, str(row["config_id"])))
    return options


def default_recompute_configs(system_id: str) -> tuple[str, ...]:
    choices = []
    for case_name in ["DFT best", "DFT-relaxed final SP best", "Toolkit-relaxed best"]:
        try:
            choices.append(str(pick_case(system_id, case_name)["config_id"]))
        except Exception:
            pass
    # Preserve order and remove duplicates.
    return tuple(dict.fromkeys(choices))


def subset_root_name(system_id: str, config_ids: list[str]) -> str:
    stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    suffix = "_".join(config_ids[:4])
    if len(config_ids) > 4:
        suffix += f"_plus{len(config_ids) - 4}"
    return f"{system_id}_{suffix}_{stamp}"


def build_subset_commands(run_root: Path, system_id: str, config_ids: list[str], *, n_steps: int, fmax: float) -> list[list[str]]:
    py = sys.executable
    scripts = PART1 / "scripts"
    chunk_size = str(max(1, min(len(config_ids), 4)))
    return [
        [
            py,
            str(scripts / "run_oc20dense_known_examples.py"),
            "--systems",
            system_id,
            "--config-ids",
            *config_ids,
            "--outdir",
            str(run_root),
            "--chunk-size",
            chunk_size,
            "--n-steps",
            str(n_steps),
            "--fmax",
            str(fmax),
            "--force",
        ],
        [
            py,
            str(scripts / "oc20dense_dft_reference_checks.py"),
            "--toolkit-root",
            str(run_root),
            "--outdir",
            str(run_root / "dft_reference_checks"),
            "--extract-dir",
            str(run_root / "dft_reference_checks" / "extracted_trajectories"),
            "--systems",
            system_id,
            "--scope",
            "all",
        ],
        [
            py,
            str(scripts / "run_oc20dense_dft_final_single_points.py"),
            "--toolkit-root",
            str(run_root),
            "--dft-check-dir",
            str(run_root / "dft_reference_checks"),
            "--outdir",
            str(run_root / "dft_final_single_points"),
            "--systems",
            system_id,
            "--chunk-size",
            chunk_size,
            "--force",
        ],
        [
            py,
            str(scripts / "run_oc20dense_mace_adsorption_energies.py"),
            "--toolkit-root",
            str(run_root),
            "--outdir",
            str(run_root / "mace_adsorption_energy"),
            "--surface-dir",
            str(run_root / "mace_adsorption_energy" / "surface_trajectories"),
            "--systems",
            system_id,
            "--n-steps",
            str(n_steps),
            "--fmax",
            str(fmax),
            "--force",
        ],
        [
            py,
            str(scripts / "summarize_oc20dense_accuracy.py"),
            "--root",
            str(run_root),
        ],
    ]


def run_command(command: list[str], *, cwd: Path) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(shlex.quote(part) for part in command), flush=True)
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")
    return completed


def run_subset_recompute(system_id: str, config_ids: list[str], *, n_steps: int = 200, fmax: float = 0.05) -> Path:
    if not config_ids:
        raise ValueError("Select at least one config_id.")
    run_root = SUBSET_BASE / subset_root_name(system_id, config_ids)
    run_root.mkdir(parents=True, exist_ok=True)
    commands = build_subset_commands(run_root, system_id, config_ids, n_steps=n_steps, fmax=fmax)
    print(f"Subset output root: {run_root}", flush=True)
    for command in commands:
        run_command(command, cwd=PART1.parent)
    return run_root

In [ ]:
def compare_subset_to_saved(run_root: Path) -> pd.DataFrame:
    fresh_pc = pd.read_csv(run_root / "tables" / "per_config_results.csv")
    fresh_dft_final = pd.read_csv(run_root / "dft_final_single_points" / "tables" / "dft_final_sp_results.csv")
    fresh_eads = pd.read_csv(run_root / "mace_adsorption_energy" / "tables" / "mace_adsorption_energies.csv")
    fresh_dft_ref = pd.read_csv(run_root / "dft_reference_checks" / "dft_reference_comparison.csv")

    key = ["system_id", "config_id", "sid"]
    comparisons = []

    def add_comparison(name: str, fresh: pd.DataFrame, saved: pd.DataFrame, columns: list[str]):
        merged = fresh[key + columns].merge(saved[key + columns], on=key, suffixes=("_fresh", "_saved"), how="left")
        saved_cols = [f"{column}_saved" for column in columns]
        merged["saved_match"] = merged[saved_cols].notna().all(axis=1)
        for column in columns:
            fresh_col = f"{column}_fresh"
            saved_col = f"{column}_saved"
            merged[f"delta_{column}"] = pd.to_numeric(merged[fresh_col]) - pd.to_numeric(merged[saved_col])
        for _, row in merged.iterrows():
            for column in columns:
                comparisons.append(
                    {
                        "check": name,
                        "system_id": row["system_id"],
                        "config_id": row["config_id"],
                        "sid": int(row["sid"]),
                        "quantity": column,
                        "fresh": row[f"{column}_fresh"],
                        "saved": row[f"{column}_saved"],
                        "delta": row[f"delta_{column}"],
                        "saved_match": bool(row["saved_match"]),
                    }
                )

    add_comparison(
        "initial_and_relaxed_toolkit",
        fresh_pc,
        per_config,
        [
            "ml_initial_sp_total_energy_eV",
            "ml_total_energy_eV",
            "optimizer_nsteps",
            "free_fmax_eV_A",
        ],
    )
    add_comparison(
        "dft_final_single_point",
        fresh_dft_final,
        dft_final_sp,
        [
            "mace_dft_final_sp_total_energy_eV",
            "mace_dft_final_sp_free_fmax_eV_A",
        ],
    )
    add_comparison(
        "defined_mace_eads",
        fresh_eads,
        tables["mace_eads"],
        [
            "mace_dft_final_eads_eV",
            "mace_relaxed_eads_eV",
        ],
    )
    add_comparison(
        "dft_reference_arithmetic",
        fresh_dft_ref,
        dft_reference,
        [
            "dft_adsorption_energy_from_traj_eV",
            "dft_adsorption_energy_target_eV",
            "dft_traj_minus_target_eV",
            "mic_active_atom_rmsd_A",
            "mic_adsorbate_rmsd_A",
        ],
    )

    comparison = pd.DataFrame(comparisons)
    comparison["abs_delta"] = pd.to_numeric(comparison["delta"]).abs()
    tolerances = {
        "ml_initial_sp_total_energy_eV": 1e-3,
        "ml_total_energy_eV": 1e-3,
        "optimizer_nsteps": 0.0,
        "free_fmax_eV_A": 5e-2,
        "mace_dft_final_sp_total_energy_eV": 1e-3,
        "mace_dft_final_sp_free_fmax_eV_A": 1e-4,
        "mace_dft_final_eads_eV": 1e-3,
        "mace_relaxed_eads_eV": 1e-3,
        "dft_adsorption_energy_from_traj_eV": 1e-12,
        "dft_adsorption_energy_target_eV": 1e-12,
        "dft_traj_minus_target_eV": 1e-12,
        "mic_active_atom_rmsd_A": 1e-2,
        "mic_adsorbate_rmsd_A": 1e-2,
    }
    comparison["tolerance"] = comparison["quantity"].map(tolerances).fillna(1e-6)
    comparison["status"] = np.select(
        [~comparison["saved_match"], comparison["abs_delta"] <= comparison["tolerance"]],
        ["MISSING_SAVED", "PASS"],
        default="REVIEW",
    )
    return comparison.sort_values(["status", "check", "system_id", "config_id", "quantity"])


def summarize_subset_run(run_root: Path) -> None:
    comparison = compare_subset_to_saved(run_root)
    display(Markdown(f"### Fresh subset comparison\n\nOutput root: `{run_root}`"))
    display(comparison)
    max_delta = (
        comparison.groupby(["check", "status"], as_index=False)["abs_delta"]
        .max()
        .sort_values(["status", "check"])
    )
    display(Markdown("Max absolute deltas against saved full-run tables"))
    display(max_delta)
    review = comparison[comparison["status"] == "REVIEW"]
    if not review.empty:
        display(Markdown("Rows marked `REVIEW` exceeded the conservative notebook tolerance. Inspect whether this is from a relaxation rerun, force noise, or a real mismatch."))
        display(review)

    subset_report = run_root / "reports" / "oc20dense_accuracy_comparison_report.md"
    if subset_report.exists():
        display(Markdown(f"Subset report: `{subset_report}`"))
    return None

In [ ]:
subset_system_dropdown = widgets.Dropdown(
    options=[(f"{sid} {ads} -> {ref}", sid) for sid, (ads, ref) in EXPECTED_SYSTEMS.items()],
    value="3_2070_48",
    description="System",
    layout=widgets.Layout(width="420px"),
)
subset_config_select = widgets.SelectMultiple(
    options=config_options_for_system("3_2070_48"),
    value=default_recompute_configs("3_2070_48"),
    description="Configs",
    rows=8,
    layout=widgets.Layout(width="620px"),
)
subset_steps = widgets.IntText(value=200, description="Max steps", layout=widgets.Layout(width="180px"))
subset_fmax = widgets.FloatText(value=0.05, description="fmax", layout=widgets.Layout(width="180px"))
subset_run_button = widgets.Button(description="Recompute selected subset", button_style="warning")
subset_command_preview = widgets.Output()
subset_run_output = widgets.Output()
LAST_SUBSET_ROOT = None


def refresh_subset_configs(*_):
    options = config_options_for_system(subset_system_dropdown.value)
    defaults = default_recompute_configs(subset_system_dropdown.value)
    subset_config_select.options = options
    subset_config_select.value = tuple(value for value in defaults if value in {opt[1] for opt in options})
    refresh_command_preview()


def refresh_command_preview(*_):
    with subset_command_preview:
        clear_output(wait=True)
        config_ids = list(subset_config_select.value)
        if not config_ids:
            print("Select at least one config_id.")
            return
        preview_root = SUBSET_BASE / f"PREVIEW_{subset_system_dropdown.value}"
        commands = build_subset_commands(
            preview_root,
            subset_system_dropdown.value,
            config_ids,
            n_steps=int(subset_steps.value),
            fmax=float(subset_fmax.value),
        )
        print("Commands that will run, with PREVIEW replaced by a timestamped output directory:\n")
        for command in commands:
            print(" ".join(shlex.quote(part) for part in command))


def on_subset_run(_):
    global LAST_SUBSET_ROOT
    with subset_run_output:
        clear_output(wait=True)
        config_ids = list(subset_config_select.value)
        LAST_SUBSET_ROOT = run_subset_recompute(
            subset_system_dropdown.value,
            config_ids,
            n_steps=int(subset_steps.value),
            fmax=float(subset_fmax.value),
        )
        summarize_subset_run(LAST_SUBSET_ROOT)

subset_system_dropdown.observe(refresh_subset_configs, names="value")
subset_config_select.observe(refresh_command_preview, names="value")
subset_steps.observe(refresh_command_preview, names="value")
subset_fmax.observe(refresh_command_preview, names="value")
subset_run_button.on_click(on_subset_run)
refresh_command_preview()

display(
    widgets.VBox(
        [
            widgets.HBox([subset_system_dropdown, subset_steps, subset_fmax]),
            subset_config_select,
            subset_run_button,
            subset_command_preview,
            subset_run_output,
        ]
    )
)

## Static structure checks

The numeric ranking is only useful if the structures still make chemical sense. The contact sheet below compares selected official DFT-relaxed final structures with Toolkit-relaxed structures rendered outside the notebook. Use it as a quick visual check before opening the interactive widgets.


In [ ]:
CONTACT_SHEET = PART1 / "outputs" / "ovito_dft_toolkit_pairs" / "dft_toolkit_selected_contact_sheet.png"
if CONTACT_SHEET.exists():
    display(Image(filename=str(CONTACT_SHEET)))
    print(f"Displayed: {CONTACT_SHEET}")
else:
    display(Markdown(f"Static OVITO contact sheet not found: `{CONTACT_SHEET}`. Run `scripts/run_oc20dense_ovito_render_pipeline.sh` to regenerate it."))


In [ ]:
def ovito_preflight() -> bool:
    try:
        import ovito
        import ovito.gui
        import ipywidgets
        from ovito.io import import_file
        from ovito.vis import Viewport
        if not hasattr(ovito.gui, "create_ipywidget"):
            raise RuntimeError("ovito.gui.create_ipywidget is unavailable")
    except Exception as exc:
        display(
            Markdown(
                "### OVITO widget preflight failed\n\n"
                f"`{type(exc).__name__}: {exc}`\n\n"
                "The numeric checks above do not depend on OVITO. For structure widgets, run this notebook "
                "with a kernel where OVITO can import and where the Jupyter front end can load `ipywidgets` "
                "and the `jupyter-ovito` widget extension. On this shell, the known failure mode is a missing "
                "OpenGL runtime such as `libOpenGL.so.0`."
            )
        )
        return False

    display(Markdown(f"### OVITO widget preflight passed\n\nOVITO version: `{ovito.version_string}`"))
    return True

OVITO_AVAILABLE = ovito_preflight()

In [ ]:
from html import escape
from ase.io import read as ase_read

DISPLAY_RADII = {
    "H": 0.32,
    "C": 0.55,
    "N": 0.58,
    "O": 0.58,
    "Cu": 0.82,
    "Au": 0.88,
    "Hg": 0.90,
    "Sr": 1.05,
    "Ca": 0.95,
    "In": 0.86,
    "P": 0.70,
    "K": 1.08,
}


def focus_from_structure(path: str | Path) -> tuple[np.ndarray, float]:
    atoms = ase_read(path)
    positions = atoms.get_positions()
    tags = np.asarray(atoms.get_tags())
    if len(tags) == len(positions) and np.any(tags == 2):
        focus = positions[tags == 2].mean(axis=0)
    elif len(tags) == len(positions) and np.any(tags > 0):
        focus = positions[tags > 0].mean(axis=0)
    else:
        focus = positions.mean(axis=0)
    radius = float(np.linalg.norm(positions - focus, axis=1).max())
    return focus, max(radius, 4.0)


def camera_for_structure(path: str | Path) -> tuple[tuple[float, float, float], tuple[float, float, float], float]:
    focus, radius = focus_from_structure(path)
    camera_unit = np.asarray([0.78, -1.18, 0.62], dtype=float)
    camera_unit /= np.linalg.norm(camera_unit)
    fov = math.radians(33.0)
    distance = max(10.0, radius / math.sin(fov / 2.0) * 1.08)
    camera_pos = focus + camera_unit * distance
    camera_dir = focus - camera_pos
    return tuple(camera_pos), tuple(camera_dir), fov


def style_ovito_data(frame, data):
    from ovito.vis import ParticlesVis

    particles = data.particles_
    particles.vis.shape = ParticlesVis.Shape.Sphere
    particles.vis.radius = 0.62
    try:
        types = particles.particle_types_
        for particle_type in types.types:
            name = str(particle_type.name)
            try:
                particle_type.load_defaults()
            except Exception:
                pass
            if name in DISPLAY_RADII:
                particle_type.radius = DISPLAY_RADII[name]
    except Exception:
        pass

    if data.cell is not None:
        data.cell_.vis.enabled = True
        data.cell_.vis.render_cell = True
        data.cell_.vis.line_width = 0.045
        data.cell_.vis.rendering_color = (0.56, 0.64, 0.61)


def make_ovito_widget(path: str | Path, title: str, width: str = "500px", height: str = "430px"):
    import ovito.gui
    import ipywidgets as widgets
    from ovito.io import import_file
    from ovito.vis import Viewport

    path = Path(str(path))
    if not path.exists():
        return widgets.HTML(f"<b>{escape(title)}</b><br><code>missing: {escape(str(path))}</code>")

    pipeline = import_file(str(path))
    pipeline.modifiers.append(style_ovito_data)
    widget = ovito.gui.create_ipywidget(
        pipeline,
        picking=True,
        layout=widgets.Layout(width=width, height=height, border="1px solid #c7cbd1"),
    )
    camera_pos, camera_dir, fov = camera_for_structure(path)
    widget.viewport.type = Viewport.Type.Perspective
    widget.viewport.camera_pos = camera_pos
    widget.viewport.camera_dir = camera_dir
    widget.viewport.fov = fov
    widget.refresh()
    return widgets.VBox(
        [
            widgets.HTML(f"<b>{escape(title)}</b><br><code>{escape(path.name)}</code>"),
            widget,
        ],
        layout=widgets.Layout(width=width),
    )

In [ ]:
ovito_output = widgets.Output()
render_button = widgets.Button(description="Open OVITO widgets", button_style="primary")


def show_ovito_widgets(_=None):
    with ovito_output:
        clear_output(wait=True)
        if not OVITO_AVAILABLE:
            display(Markdown("OVITO did not pass preflight in this kernel. The selected paths above are still available for manual inspection."))
            return
        row = pick_case(system_dropdown.value, case_dropdown.value)
        initial_panel = make_ovito_widget(row["initial_structure"], "OC20Dense initial geometry")
        dft_final_panel_for_initial_tab = make_ovito_widget(row["dft_final_structure_path"], "Official DFT final geometry")
        dft_final_panel_for_relaxed_tab = make_ovito_widget(row["dft_final_structure_path"], "Official DFT final geometry")
        relaxed_panel = make_ovito_widget(row["relaxed_structure"], "Toolkit-relaxed geometry")

        tabs = widgets.Tab()
        tabs.children = [
            widgets.HBox([initial_panel, dft_final_panel_for_initial_tab], layout=widgets.Layout(gap="12px")),
            widgets.HBox([dft_final_panel_for_relaxed_tab, relaxed_panel], layout=widgets.Layout(gap="12px")),
        ]
        tabs.set_title(0, "Initial vs DFT")
        tabs.set_title(1, "DFT vs Toolkit")
        display(case_summary(row))
        display(tabs)

render_button.on_click(show_ovito_widgets)
display(render_button, ovito_output)

## Optional rebuild commands

Run these from the repository root if you want to regenerate the current outputs before reopening this notebook. They are not executed automatically here.

In [ ]:
REBUILD_ROOT = "part-1-batched-adsorption/outputs/live_runs/manual/accuracy/oc20dense_rebuild"
REBUILD_COMMANDS = [
    f".venv-toolkit/bin/python part-1-batched-adsorption/scripts/run_oc20dense_known_examples.py --outdir {REBUILD_ROOT}",
    f".venv-toolkit/bin/python part-1-batched-adsorption/scripts/oc20dense_dft_reference_checks.py --toolkit-root {REBUILD_ROOT} --outdir {REBUILD_ROOT}/dft_reference_checks --mode compare --scope all",
    f".venv-toolkit/bin/python part-1-batched-adsorption/scripts/run_oc20dense_dft_final_single_points.py --toolkit-root {REBUILD_ROOT} --dft-check-dir {REBUILD_ROOT}/dft_reference_checks --outdir {REBUILD_ROOT}/dft_final_single_points --force",
    f".venv-toolkit/bin/python part-1-batched-adsorption/scripts/run_oc20dense_mace_adsorption_energies.py --toolkit-root {REBUILD_ROOT} --dft-check-dir {REBUILD_ROOT}/dft_reference_checks --dft-final-sp-dir {REBUILD_ROOT}/dft_final_single_points --outdir {REBUILD_ROOT}/mace_adsorption_energy",
    ".venv-toolkit/bin/python part-1-batched-adsorption/scripts/summarize_oc20dense_accuracy.py",
    ".venv-toolkit/bin/python -m pytest -q part-1-batched-adsorption/tests/test_oc20dense_benchmark.py",
]
print("\n".join(REBUILD_COMMANDS))